# Práctica: LDA vs QDA vs RDA

Objetivo:
- Trabajar con Iris para clasificar solo Versicolor y Virginica.
- Usar únicamente sepal length y sepal width.
- Comparar LDA, QDA y una aproximación a RDA usando shrinkage en LDA.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix

## 1) Importar y filtrar los datos

- Cargamos Iris.
- Nos quedamos solo con Versicolor (1) y Virginica (2).
- Usamos solo sepal length (col 0) y sepal width (col 1).
- Comprobamos cuántas observaciones quedan.

In [ ]:
iris = load_iris()

X_all = iris.data
y_all = iris.target
target_names = iris.target_names

mask = (y_all == 1) | (y_all == 2)
X = X_all[mask][:, [0, 1]]   # sepal length, sepal width
y = y_all[mask]

print("Clases usadas:", target_names[1], "y", target_names[2])
print("Observaciones tras filtrado:", X.shape[0])
print("Dimensión X:", X.shape)

df = pd.DataFrame(X, columns=["sepal_length", "sepal_width"])
df["class"] = y
df.head()

## 2) División entrenamiento / test

- Usamos train_test_split.
- test_size=0.3 y random_state=42 para reproducibilidad.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

## 3) Entrenar LDA, QDA y “RDA” (shrinkage en LDA)

Modelos:
- LDA: LinearDiscriminantAnalysis()
- QDA: QuadraticDiscriminantAnalysis()
- RDA(0.25), RDA(0.5), RDA(0.75): LinearDiscriminantAnalysis(solver='lsqr', shrinkage=alpha)

In [ ]:
lda = LinearDiscriminantAnalysis(store_covariance=True)
qda = QuadraticDiscriminantAnalysis()

# RDA aproximado con shrinkage en LDA (solver='lsqr')
rda_025 = LinearDiscriminantAnalysis(solver='lsqr', shrinkage=0.25)
rda_050 = LinearDiscriminantAnalysis(solver='lsqr', shrinkage=0.50)
rda_075 = LinearDiscriminantAnalysis(solver='lsqr', shrinkage=0.75)

models = {
    "LDA": lda,
    "QDA": qda,
    "RDA(0.25)": rda_025,
    "RDA(0.5)": rda_050,
    "RDA(0.75)": rda_075
}

for name, model in models.items():
    model.fit(X_train, y_train)

## ¿Por qué no usamos RDA(0) o RDA(1)?

- Con shrinkage=0 no hay regularización: equivale a LDA “normal” (mismo resultado que LDA).
- Con shrinkage=1 la regularización es máxima: la covarianza se fuerza demasiado hacia una forma muy simple, pudiendo perder información útil y empeorar el ajuste.
Por eso probamos valores intermedios para ajustar la complejidad (0.25, 0.5, 0.75).

## 4) Evaluación: accuracy y matriz de confusión

- Para cada modelo: predict sobre X_test.
- Calculamos accuracy_score y confusion_matrix.
- Mostramos una tabla resumen.

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=[1,2])
    results.append({"model": name, "accuracy": acc, "confusion_matrix": cm})

df_results = pd.DataFrame([{"model": r["model"], "accuracy": r["accuracy"]} for r in results])
df_results = df_results.sort_values("accuracy", ascending=False).reset_index(drop=True)

print("Tabla de accuracies (test):")
display(df_results)

print("Matrices de confusión (filas=real, columnas=pred): labels=[Versicolor(1), Virginica(2)]")
for r in results:
    print("\n", r["model"])
    print(r["confusion_matrix"])

## 5) Visualización de fronteras de decisión

- Creamos un grid 2D sobre el rango de sepal_length y sepal_width.
- Para cada punto, predecimos la clase y pintamos la región.
- Mostramos subplots:
  (1) LDA, (2) QDA, (3) RDA(0.5)
- También se dibuja un scatter de X_train con colores según la clase.

In [ ]:
# Crear grid
x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]

def plot_boundary(ax, model, title):
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3)
    ax.scatter(X_train[:,0], X_train[:,1], c=y_train, s=18)
    ax.set_title(title)
    ax.set_xlabel("sepal length")
    ax.set_ylabel("sepal width")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

plot_boundary(axes[0], lda, "LDA")
plot_boundary(axes[1], qda, "QDA")
plot_boundary(axes[2], rda_050, "RDA(0.5)")

plt.tight_layout()
plt.show()

## 6) Visualización extra: ver RDAs con distintos shrinkage

Mostramos en una figura las fronteras de:
- RDA(0.25)
- RDA(0.5)
- RDA(0.75)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

plot_boundary(axes[0], rda_025, "RDA(0.25)")
plot_boundary(axes[1], rda_050, "RDA(0.5)")
plot_boundary(axes[2], rda_075, "RDA(0.75)")

plt.tight_layout()
plt.show()

In [ ]:
# Esto permite comparar: en LDA se asume una covarianza común (una sola matriz),
# mientras que en QDA se permite una covarianza distinta por clase.
# Aquí calculo covarianzas a mano a partir de los datos para ver esa diferencia.

print("Covarianza 'pooled' (común) estimada a mano (para idea LDA):")
cov_pooled = np.cov(X_train.T, bias=False)
print(cov_pooled)

print("\nCovarianzas por clase estimadas a mano (idea QDA):")
for c in np.unique(y_train):
    Xc = X_train[y_train == c]
    cov_c = np.cov(Xc.T, bias=False)
    print("Clase", c, "->")
    print(cov_c)

## 7) Comentarios - Resumen

Forma de las fronteras:
- LDA: frontera lineal porque asume una covarianza común para ambas clases.
- QDA: frontera curva porque estima una covarianza distinta por clase.
- RDA: al regularizar la covarianza, la frontera queda “entre” LDA y QDA (según alpha). Si aumentamos el shrinkage, el modelo se vuelve más rígido y la frontera tiende a parecer más simple.

Comparación de exactitud:
- En mis resultados, el mejor rendimiento en test lo obtiene RDA (0.25/0.5/0.75), que en este caso dan el mismo accuracy: 0.7333.
- LDA obtiene 0.7000 y QDA 0.6667. Esto sugiere que QDA, al estimar una covarianza por clase (más parámetros), puede tener más varianza y generalizar peor en este split, mientras que RDA reduce esa varianza al regularizar.

Matrices de confusión (filas=real, columnas=pred; orden [Versicolor(1), Virginica(2)]):
- LDA: 2 errores en Versicolor y 7 en Virginica (accuracy 0.7000).
- QDA: 3 errores en Versicolor y 7 en Virginica (accuracy 0.6667).
- RDA: 1 error en Versicolor y 7 en Virginica (accuracy 0.7333).

Covarianzas y efecto de la regularización:
- En QDA cada clase tiene su propia covarianza, lo que da más flexibilidad y explica una frontera curva.
- En LDA se usa una única covarianza “común”, por eso la frontera es lineal.
- Con shrinkage (RDA) se regulariza esa covarianza: la frontera se parece más a la de LDA, pero en este caso generaliza mejor que LDA y QDA.